# 03 - Error analysis

**Goal:** find out *where and why* the model gets Hinglish wrong. This is the part that makes the project more than a table of numbers.

Needs the MuRIL model pushed to the Hugging Face Hub (from notebook 02). Set `HF_USER` below.

In [ ]:
%pip install -q ftfy tweet-preprocessor

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/23f2004336/hinglish-sentiment.git"

# Make `src/` importable: works locally (inside notebooks/ or the repo root) and on Kaggle.
if os.path.exists("../src"):
    sys.path.insert(0, "..")
    RESULTS_DIR = "../results"
elif os.path.exists("src"):
    sys.path.insert(0, ".")
    RESULTS_DIR = "results"
else:
    if not os.path.exists("hinglish-sentiment"):
        subprocess.run(["git", "clone", REPO_URL], check=True)   # needs Internet ON in Kaggle
    sys.path.insert(0, os.path.abspath("hinglish-sentiment"))
    RESULTS_DIR = "."

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score
from transformers import pipeline
from src.data import load_data

HF_USER = "YOUR-HF-USERNAME"      # <-- change this
REPO_ID = f"{HF_USER}/hinglish-sentiment-muril"
NAMES = {0: "negative", 1: "neutral", 2: "positive"}

dfs = load_data()
test = dfs["test"].copy()

## Predict on the test set with the saved model

In [ ]:
clf = pipeline(
    "text-classification", model=REPO_ID, tokenizer=REPO_ID,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True, max_length=128, batch_size=64,
)
outs = clf(test.text.tolist())
test["pred"] = [int(o["label"].split("_")[-1]) for o in outs]

y, p, m = test.label.values, test.pred.values, test.hinglish.values
print("Overall :", round(f1_score(y, p, average="macro"), 3))
print("Hinglish:", round(f1_score(y[m], p[m], average="macro"), 3))
print("English :", round(f1_score(y[~m], p[~m], average="macro"), 3))

## Confusion matrix (Hinglish only)

In [ ]:
h = test[test.hinglish].copy()
cm = confusion_matrix(h.label, h.pred)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
labels = [NAMES[i] for i in range(3)]
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title("Hinglish test slice")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## Look at the mistakes
Read these carefully. Look for sarcasm, abuse used casually, very short texts, spelling variants, and labels that look wrong.

In [ ]:
def show(true, predicted, n=8):
    rows = h[(h.label == true) & (h.pred == predicted)]
    print(f"\n=== true {NAMES[true]} -> predicted {NAMES[predicted]}  ({len(rows)} rows) ===")
    for t in rows.text.head(n):
        print("-", t[:160])

show(0, 1)   # negative predicted as neutral
show(2, 1)   # positive predicted as neutral
show(1, 0)   # neutral predicted as negative
show(1, 2)   # neutral predicted as positive

## Does text length matter?

In [ ]:
h["n_words"] = h.text.str.split().str.len()
h["bucket"] = pd.cut(h.n_words, [0, 5, 10, 20, 1000], labels=["<=5", "6-10", "11-20", "20+"])
h["correct"] = h.label == h.pred
h.groupby("bucket", observed=True).correct.agg(["mean", "size"]).round(2)

## Findings (write these yourself after reading the examples)

- **Main pattern:** ...
- **Positive class:** ... (recall was about 0.40)
- **Negative class:** ...
- **Label noise:** examples where the label itself looks wrong: ...
- **What I would try next:** ... (more Hinglish data, oversampling, class weights, better tagging)